Harry Ferguson - CIS 361 - Summer 2026 - Assignment 8

In [9]:
import numpy as np
import pandas as pd
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense
from tensorflow.keras.models import load_model

df = pd.read_table('lit-eng/lit.txt', header=None, names=['E','L','X'])
df.head()

english = df['E'].tolist()
lithuanian = df['L'].tolist()

print('Number of English sentences: ', len(english))
print('Number of Lithuanian sentences: ', len(lithuanian))

lithuanian = ['<start>' + text + '<eend>' for text in lithuanian]

def mapper(texts):
    chars = sorted(set(' '.join(texts)))
    return chars, len(chars), dict((c, i) for i, c in enumerate(chars)), max([len(txt) for txt in texts])

english_chars, num_encoder_tokens, english_token_index, max_encoder_seq_length = mapper(english)
lithuanian_chars, num_decoder_tokens, lithuanian_token_index, max_decoder_seq_length = mapper(lithuanian)

num_decoder_tokens += 2  # Increase the number of decoder tokens to accommodate the special tokens
lithuanian_token_index['<start>'] = num_decoder_tokens - 2
lithuanian_token_index['<end>'] = num_decoder_tokens - 1

reverse_english_char_index = dict((i, c) for c, i in english_token_index.items())
reverse_lithuanian_char_index = dict((i, c) for c, i in lithuanian_token_index.items())

print(english_token_index)
print(lithuanian_token_index)

encoder_input_data = np.zeros((len(english), max_encoder_seq_length, num_encoder_tokens), dtype='float32')
decoder_input_data = np.zeros((len(english), max_decoder_seq_length, num_decoder_tokens), dtype='float32')
decoder_target_data = np.zeros((len(english), max_decoder_seq_length, num_decoder_tokens), dtype='float32')

for i, (eng, ger) in enumerate(zip(english, lithuanian)):
    for t, char in enumerate(eng):
        encoder_input_data[i, t, english_token_index[char]] = 1.0
    for t, char in enumerate(ger):
        decoder_input_data[i, t, lithuanian_token_index[char]] = 1.0
        if t > 0:
            decoder_target_data[i, t - 1, lithuanian_token_index[char]] = 1.0

Number of English sentences:  2290
Number of Lithuanian sentences:  2290
{' ': 0, '!': 1, '"': 2, "'": 3, ',': 4, '-': 5, '.': 6, '0': 7, '1': 8, '2': 9, '3': 10, '5': 11, '6': 12, '7': 13, '8': 14, '9': 15, ':': 16, '?': 17, 'A': 18, 'B': 19, 'C': 20, 'D': 21, 'E': 22, 'F': 23, 'G': 24, 'H': 25, 'I': 26, 'J': 27, 'K': 28, 'L': 29, 'M': 30, 'N': 31, 'O': 32, 'P': 33, 'R': 34, 'S': 35, 'T': 36, 'U': 37, 'V': 38, 'W': 39, 'Y': 40, 'Z': 41, 'a': 42, 'b': 43, 'c': 44, 'd': 45, 'e': 46, 'f': 47, 'g': 48, 'h': 49, 'i': 50, 'j': 51, 'k': 52, 'l': 53, 'm': 54, 'n': 55, 'o': 56, 'p': 57, 'q': 58, 'r': 59, 's': 60, 't': 61, 'u': 62, 'v': 63, 'w': 64, 'x': 65, 'y': 66, 'z': 67}
{' ': 0, '!': 1, '"': 2, ',': 3, '-': 4, '.': 5, '0': 6, '1': 7, '2': 8, '3': 9, '5': 10, '8': 11, '9': 12, ':': 13, '<': 14, '>': 15, '?': 16, 'A': 17, 'B': 18, 'C': 19, 'D': 20, 'E': 21, 'F': 22, 'G': 23, 'H': 24, 'I': 25, 'J': 26, 'K': 27, 'L': 28, 'M': 29, 'N': 30, 'O': 31, 'P': 32, 'R': 33, 'S': 34, 'T': 35, 'U': 36, 

In [10]:
# Define the model
latent_dim = 256

# Define encoder
encoder_inputs = Input(shape=(None, num_encoder_tokens))
encoder = LSTM(latent_dim, return_state=True)
encoder_outputs, state_h, state_c = encoder(encoder_inputs)
encoder_states = [state_h, state_c]

# Define decoder
decoder_inputs = Input(shape=(None, num_decoder_tokens))
decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(decoder_inputs, initial_state=encoder_states)
decoder_dense = Dense(num_decoder_tokens, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

In [ ]:
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer='rmsprop', loss='categorical_crossentropy', metrics=['accuracy'])
model.fit([encoder_input_data, decoder_input_data], decoder_target_data, batch_size=64, epochs=100, validation_split=0.2)
model.save('eng_lit_model.h5')

Epoch 1/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 21s 674ms/step - accuracy: 0.1845 - loss: 0.8935 - val_accuracy: 0.0629 - val_loss: 1.2677
Epoch 2/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 19s 645ms/step - accuracy: 0.0422 - loss: 0.7563 - val_accuracy: 0.0580 - val_loss: 1.2455
Epoch 3/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 20s 676ms/step - accuracy: 0.0489 - loss: 0.7347 - val_accuracy: 0.0740 - val_loss: 1.2188
Epoch 4/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 20s 685ms/step - accuracy: 0.0587 - loss: 0.7234 - val_accuracy: 0.0753 - val_loss: 1.1994
Epoch 5/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 19s 646ms/step - accuracy: 0.0589 - loss: 0.7150 - val_accuracy: 0.0728 - val_loss: 1.2121
Epoch 6/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 19s 673ms/step - accuracy: 0.0594 - loss: 0.7068 - val_accuracy: 0.0730 - val_loss: 1.2060
Epoch 7/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 18s 637ms/step - accuracy: 0.0595 - loss: 0.7002 - val_accuracy: 0.0743 - val_loss: 1.1919
Epoch 8/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 19s 645ms/step - accuracy: 0.0593 - loss: 0.6945 - 

In [15]:
model = load_model('eng_lit_model.h5')

encoder_model = Model(encoder_inputs, encoder_states)

decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

decoder_outputs, state_h, state_c = decoder_lstm(decoder_inputs, initial_state=decoder_states_inputs)
decoder_states = [state_h, state_c]
decoder_outputs = decoder_dense(decoder_outputs)

decoder_model = Model([decoder_inputs] + decoder_states_inputs, [decoder_outputs] + decoder_states)

def translate_sentence(input_text):
    input_seq = np.zeros((1, max_encoder_seq_length, num_encoder_tokens), dtype='float32')
    for t, char in enumerate(input_text):
        input_seq[0, t, english_token_index[char]] = 1.0

    states_value = encoder_model.predict(input_seq)

    target_seq = np.zeros((1, 1, num_decoder_tokens), dtype='float32')
    target_seq[0, 0, lithuanian_token_index['<start>']] = 1.0

    stop_condition = False
    decoded_sentence = ''
    while not stop_condition:
        output_tokens, h, c = decoder_model.predict([target_seq] + states_value)

        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_char = reverse_lithuanian_char_index[int(sampled_token_index)]
        decoded_sentence += sampled_char

        if sampled_char == '<end>' or len(decoded_sentence) > max_decoder_seq_length:
            stop_condition = True

        target_seq = np.zeros((1, 1, num_decoder_tokens), dtype='float32')
        target_seq[0, 0, sampled_token_index] = 1.0

        states_value = [h, c]

    return decoded_sentence

input_text = 'I am learning to translate.'
translated_text = translate_sentence(input_text)
print('Input:', input_text)
print('Translation:', translated_text)

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 588ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 382ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 402ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 570ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 647ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 774ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 782ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 436ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 687ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 593ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 479ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 614ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 417ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 514ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 329ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 488ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 271ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 387ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 409ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 398ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 303ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 611ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 213ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 791

In selecting a language for translation, I first choose German, due to the large number of available words and phrases (331266). However, when the code reached the one-hot encoding stage, the following error occured:

*Unable to allocate 68.9 GiB for an array with shape (331266, 537, 104) and data type float32*

I then switched to Romanian, with 16297 translations. However, the time to train the model was too long. Finally, I settled on Lithuanian with 2290 translations. Over 100 epochs of training, the accuracy of the model slowly improved, going from an accuracy score of 0.0422 in epoch 2 to 0.1068 in epoch 100. Still, the model is not very accurate, as can be seen from the translation of the English sentence "I am learning to translate". According to Google Translate, "Aš nas ai nai tai" translates to "I am not here." The model does get part of the translation right: "Aš nas" -> "I am", and with a longer training period and a larger corpus, the model could very well be an accurate translator.